In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl
import statsmodels.api as sm

# Optional für Plots
import matplotlib.pyplot as plt

In [2]:


# Excel-Datei im selben Ordner wie das Notebook:
xlsx_path = Path.cwd() / "AMLTA Experiment.xlsx"
xls = pd.ExcelFile(xlsx_path)
xls.sheet_names
sheet = "Experiment Data"

In [3]:
df_races = pd.read_excel(
    xlsx_path,
    sheet_name=sheet,
    header=4,          # Zeile 5 enthält Header
    usecols="E:K",     # nur diese Spalten
    nrows=64           # 68 - 5 + 1 = 64 Zeilen
)

df_laps = pd.read_excel(
    xlsx_path,
    sheet_name=sheet,
    header=4,
    usecols="N:U",
    nrows=316          # 320 - 5 + 1 = 316 Zeilen
)

df_races.head()

,name,id,racetrack,help,experience,race_number,total_duration
0,AmL ID5,1.0,1.0,1.0,1.0,1.0,313.02
1,AmL ID5,1.0,2.0,2.0,1.0,2.0,510.20
2,AmL ID5,1.0,3.0,3.0,1.0,3.0,497.33
3,ML ID7,2.0,2.0,2.0,1.0,1.0,464.05
4,ML ID7,2.0,3.0,3.0,1.0,2.0,476.25


In [4]:
df_races = df_races.dropna(how="all")
df_laps  = df_laps.dropna(how="all")


In [5]:
df_races["total_duration"] = (
    df_races["total_duration"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [6]:
# for each id in df_races, sum up the total_durations to calculate the three_race_duration
df_races["three_race_duration"] = df_races.groupby("id")["total_duration"].transform("sum")
df_laps["three_race_duration"] = df_laps.groupby("id.1")["duration"].transform("sum")


In [7]:
# save to csv
df_races.to_csv("csv/races.csv", index=False)
df_laps.to_csv("csv/laps.csv", index=False) 

In [8]:
hilfe_map = {
    1: "No Help",
    2: "With Driving Coach",
    3: "With Ideal Driving Line"
}

df_races["help_label"] = df_races["help"].map(hilfe_map)
df_laps["help_label"] = df_laps["help.1"].map(hilfe_map)


In [9]:
grouped = (
    df_races
    .groupby(["help_label", "racetrack"])["total_duration"]
    .mean()
    .reset_index()
)

grouped

,help_label,racetrack,total_duration
0,No Help,1.0,296.488750
1,No Help,2.0,379.925000
2,No Help,3.0,224.977143
3,With Driving Coach,1.0,173.534286
4,With Driving Coach,2.0,420.692500
5,With Driving Coach,3.0,391.995000
6,With Ideal Driving Line,1.0,284.466667
7,With Ideal Driving Line,2.0,206.738571
8,With Ideal Driving Line,3.0,406.722500


In [10]:
mean_total = (
    df_races
    .groupby("help_label")["total_duration"]
    .mean()
    .reset_index()
)

plt.figure()
plt.bar(mean_total["help_label"], mean_total["total_duration"])
plt.xlabel("Help condition")
plt.ylabel("Average total duration (s)")
plt.title("Total duration by help condition")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("pictures/total_duration_by_help_condition.png")
plt.close()

In [11]:
mean_lap = (
    df_laps
    .groupby(["help_label", "lap"])["duration"]
    .mean()
    .reset_index()
)

plt.figure()

for hilfe in mean_lap["help_label"].unique():
    subset = mean_lap[mean_lap["help_label"] == hilfe]
    plt.plot(subset["lap"], subset["duration"], marker="o", label=hilfe)

plt.xlabel("Lap number")
plt.ylabel("Average lap duration (s)")
plt.title("Lap-to-Lap Duration by Help Condition")
plt.legend()
plt.tight_layout()
plt.savefig("pictures/lap_improvement_by_help_condition.png")
plt.close()

In [12]:
# Perform one hot encoding and save to csv
df_races_encoded = pd.get_dummies(
    df_races,
    columns=["racetrack", "help", "experience", "race_number"],
    drop_first=True
)

df_laps_encoded = pd.get_dummies(
    df_laps,
    columns=["racetrack.1", "lap", "help.1", "experience.1", "race_number.1"],
    drop_first=True
)

df_races_encoded.to_csv("csv/races_oneHot.csv", index=False)
df_laps_encoded.to_csv("csv/laps_oneHot.csv", index=False) 

In [13]:
# Fit regression model
X = df_races_encoded[
        [
            "racetrack_2.0",
            "racetrack_3.0",
            "help_2.0",
            "help_3.0",
            "experience_2.0",
            "experience_3.0",
            "race_number_2.0",
            "race_number_3.0",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.282
Model:                            OLS   Adj. R-squared:                  0.176
Method:                 Least Squares   F-statistic:                     2.653
Date:                Thu, 26 Feb 2026   Prob (F-statistic):             0.0157
Time:                        22:28:01   Log-Likelihood:                -386.50
No. Observations:                  63   AIC:                             791.0
Df Residuals:                      54   BIC:                             810.3
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const             286.8356     44.979     

In [14]:
# Fit regression model
X = df_races_encoded[
        [
            "help_2.0",
            "help_3.0",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                 -0.021
Method:                 Least Squares   F-statistic:                    0.3547
Date:                Thu, 26 Feb 2026   Prob (F-statistic):              0.703
Time:                        22:28:01   Log-Likelihood:                -396.58
No. Observations:                  63   AIC:                             799.2
Df Residuals:                      60   BIC:                             805.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        296.4905     29.313     10.114      0.0

In [15]:
# Fit regression model
X = df_races_encoded[
        [
            "racetrack_2.0",
            "racetrack_3.0",
            "help_2.0",
            "help_3.0",
            # "experience_2.0",
            # "experience_3.0",
            "race_number_2.0",
            "race_number_3.0",
            "three_race_duration"
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     88.66
Date:                Thu, 26 Feb 2026   Prob (F-statistic):           1.25e-27
Time:                        22:28:01   Log-Likelihood:                -317.94
No. Observations:                  63   AIC:                             651.9
Df Residuals:                      55   BIC:                             669.0
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                 -56.2082    